# Continuity-Tracker Overlay: All MAT files together

This notebook calls `continuity_multi_ridge_tracker.py` and overlays tracked ridges from all MAT files in one figure.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from continuity_multi_ridge_tracker import (
    TrackerConfig,
    run_tracker_from_mat,
)

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_files = sorted(Path('../Result').glob('*.mat'))

# If your .mat params contain k and mx, keep these consistent with simulation settings
k_coupling = 1.0
mx = 1.0

cfg = TrackerConfig(
    n_ridges=3,
    omega_min=0.01,
    skip_transient=0.20,
    n_harmonic=4,
    pts_per_cycle=12,
    max_candidates_per_k=6,
    peak_prominence_db=2.0,
    jump_penalty=1.5,
    amplitude_weight=1.0,
    suppression_half_width_bins=2,
)

print('MAT files:')
for p in mat_files:
    print(' -', p)

In [ ]:
# Run continuity-based ridge tracking for each MAT file
results = []
for p in mat_files:
    out = run_tracker_from_mat(p, k_coupling=k_coupling, mx=mx, cfg=cfg)
    label = Path(p).stem
    if 'phi2' in out['params']:
        label += f" (phi2={out['params']['phi2']:.3g})"
    results.append((label, out))

print(f'Processed {len(results)} files.')

In [ ]:
# Plot all tracked ridges from all files together
plt.figure(figsize=(10, 6.5))

for label, out in results:
    k_plot = out['k_pos'] / np.pi
    for i, ridge_omega in enumerate(out['ridges_omega'], start=1):
        valid = np.isfinite(ridge_omega)
        if not np.any(valid):
            continue
        plt.plot(k_plot[valid], ridge_omega[valid], linewidth=2, label=f"{label} - Ridge {i}")

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title('Continuity-Based Multi-Ridge Overlay (All MAT files)')
plt.grid(alpha=0.25)
plt.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()